# 11785 Project Group 30

## Baseline Running

#### Download Github & Model & Dataset

In [ ]:
!git clone https://github.com/MaoliYulin/11785_Project.git

In [ ]:
%cd /content/11785_Project/

In [ ]:
!pip install -U gdown
!gdown --fuzzy "https://drive.google.com/file/d/1vOFcavKS7u3B-pFNWAxXMJyP_voXXDpq/view?usp=drive_link" -O test_images.zip
!unzip test_images.zip

#### Download pre-train

In [ ]:
!export TORCH_HOME=$(pwd) && export PYTHONPATH=$(pwd)

In [ ]:
!curl -LJO https://huggingface.co/smartywu/big-lama/resolve/main/big-lama.zip
!unzip big-lama.zip

#### Environment Setting

In [ ]:
# this will restart the session
%pip uninstall -y jax jaxlib umap-learn cuml-cu12 tsfresh
%pip install -U \
  "numpy==1.26.4" \
  "scipy==1.14.1" \
  "scikit-learn==1.6.1" \
  "pillow==11.1.0" \
  "opencv-python-headless==4.10.0.84" \
  "imgaug==0.4.0" \
  "scikit-image==0.22.0" \
  webdataset omegaconf pyyaml easydict joblib tqdm

In [ ]:
!pip install "kornia==0.5.8"
!pip install lpips pytorch-ssim
!pip install pytorch-lightning==1.2.9
!pip install -U hydra-core==1.3.2 omegaconf==2.3.0
!pip install -U "imgaug==0.4.0" "albumentations==0.4.6"

In [ ]:
%env PYTHONPATH=/content/11785_Project:$PYTHONPATH

In [ ]:
!python -c "import saicinpainting, sys; print('OK, import saicinpainting'); print(sys.path[0])"

In [ ]:
%cd /content/11785_Project
import re, pathlib
p = pathlib.Path('saicinpainting/training/trainers/__init__.py')
txt = p.read_text()
new = re.sub(r"torch\.load\(([^)]*)\)", r"torch.load(\1, weights_only=False)", txt, count=1)
p.write_text(new)
print(" Patched:", p)

#### Visualize Photo with Mask

In [ ]:
import os
import matplotlib.pyplot as plt
import cv2
import shutil

root = "/content/11785_Project/test_gt_samples_1000/test_gt/random_medium_1024" # random_medium_1024 random_thick_1024 random_thin_1024
root_dst = "/content/11785_Project/test_gt_samples_1000/test_gt/random_medium_1024_first100" # random_medium_1024_first100 random_thick_1024_first100 random_thin_1024_first100

os.makedirs(root_dst, exist_ok=True)
files = sorted(os.listdir(root))
n = 10
number_subset = n*2
files_100_pairs = files[:number_subset]
print("len: ", len(files_100_pairs))

for f in files_100_pairs:
    shutil.copy(
        os.path.join(root, f),
        os.path.join(root_dst, f)
    )

files = sorted(os.listdir(root))
selected_input_length = 6
selected_input = files[:selected_input_length]

print("photo:", selected_input)

plt.figure(figsize=(18, 6))
for i, fname in enumerate(selected_input):
    img_path = os.path.join(root, fname)
    img = cv2.imread(img_path)[:, :, ::-1]
    plt.subplot(1, 6, i + 1)
    plt.imshow(img)
    plt.title(fname, fontsize=9)
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# sub test data
# indir = random_medium_1024_first100 random_thick_1024_first100 random_thin_1024_first100
# outdir = medium_output thick_output thin_output

!python3 bin/predict.py \
  model.path=/content/11785_Project/big-lama \
  indir=/content/11785_Project/test_gt_samples_1000/test_gt/random_medium_1024_first100 \
  outdir=/content/11785_Project/medium_output \
  hydra.run.dir=. hydra.output_subdir=null

# more test data
# random_medium_1024 random_thick_1024 random_thin_1024
# outdir = medium_output thick_output thin_output

# !python3 bin/predict.py \
#   model.path=/content/11785_Project/big-lama \
#   indir=/content/11785_Project/test_gt_samples_1000/test_gt/random_medium_1024 \
#   outdir=/content/11785_Project/medium_output \
#   hydra.run.dir=. hydra.output_subdir=null

#### Visualize Inpainting Output

In [ ]:
import os
import matplotlib.pyplot as plt
import cv2
root = "/content/11785_Project/medium_output" # medium_output thick_output thin_output
files = sorted(os.listdir(root))

number = selected_input_length//2
print("print_num: ",number)
selected = files[:number]
print("photo:", selected)

plt.figure(figsize=(18, 6))
for i, fname in enumerate(selected):
    img_path = os.path.join(root, fname)
    img = cv2.imread(img_path)[:, :, ::-1]
    plt.subplot(1, 6, i + 1)
    plt.imshow(img)
    plt.title(fname, fontsize=9)
    plt.axis('off')

plt.tight_layout()
plt.show()

#### Metrics - Learned Perceptual Image Patch Similarity & Fréchet Inception Distance

In [ ]:
!pip -q install --no-cache-dir pillow==10.4.0
!pip -q install torchmetrics==1.4.0 lpips==0.1.4 torch-fidelity==0.3.0 clean-fid==0.1.35

In [ ]:
import importlib
import torchmetrics.image.fid as fidmod
importlib.reload(fidmod)
from torchmetrics.image.fid import FrechetInceptionDistance

In [ ]:
import os, shutil
Refine_DIR = "/content/11785_Project/medium_output" # medium_output thick_output thin_output
Test_DIR = "/content/11785_Project/test_gt_samples_1000/test_gt/random_medium_1024"  # random_medium_1024 random_thick_1024 random_thin_1024
Test_NoMask_DIR = "/content/11785_Project/LaMa_test_images_nomask"
os.makedirs(Test_NoMask_DIR, exist_ok=True)

IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

def is_valid_img(fname):
    ext = os.path.splitext(fname)[1].lower()
    name = os.path.basename(fname).lower()
    return ext in IMG_EXTS and "mask" not in name

count = 0
for root, _, files in os.walk(Test_DIR):
    for f in files:
        if is_valid_img(f):
            src_path = os.path.join(root, f)
            dst_path = os.path.join(Test_NoMask_DIR, f)
            shutil.copy2(src_path, dst_path)
            count += 1

Test_DIR_dst = "/content/11785_Project/test_gt_samples_1000/test_gt/random_medium_1024_first100_gt" # random_medium_1024_first100_gt random_thick_1024_first100_gt random_thin_1024_first100_gt

os.makedirs(Test_DIR_dst, exist_ok=True)
file_s = sorted(os.listdir(Test_NoMask_DIR))

print("subset: ",number_subset)
number_subset = int(number_subset//2)
files_pairs = file_s[:number_subset]
print("len: ", len(files_pairs))

for f in files_pairs:
    shutil.copy(
        os.path.join(Test_NoMask_DIR, f),
        os.path.join(Test_DIR_dst, f)
    )

print(f" copy {len(files_pairs)} photo no mask: {Test_DIR_dst}")


In [ ]:
import os
import numpy as np
from PIL import Image
import torch
import torchvision.transforms.functional as TF
import lpips

IMG_EXTS = ('.png', '.jpg', '.jpeg', '.bmp', '.webp')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

lpips_fn = lpips.LPIPS(net='alex').to(device) # vgg alex
lpips_fn.eval()

def load_img_to_tensor(path):
    img = Image.open(path).convert('RGB')
    t = TF.to_tensor(img).unsqueeze(0)
    t = t * 2.0 - 1.0
    return t

gt_dict = {}
for fname in os.listdir(Test_DIR_dst):
    if not fname.lower().endswith(IMG_EXTS):
        continue
    key = os.path.splitext(fname)[0]
    gt_dict[key] = os.path.join(Test_DIR_dst, fname)

print(f"GT image number: {len(gt_dict)}")

lpips_values = []
missing_pairs = []

for root, _, files in os.walk(Refine_DIR):
    for fname in files:
        if not fname.lower().endswith(IMG_EXTS):
            continue

        pred_path = os.path.join(root, fname)
        pred_key  = os.path.splitext(fname)[0]
        gt_key = pred_key.replace('_mask000', '').replace('_mask', '')

        if gt_key not in gt_dict:
            missing_pairs.append(fname)
            continue

        gt_path = gt_dict[gt_key]
        pred_img = load_img_to_tensor(pred_path).to(device)
        gt_img   = load_img_to_tensor(gt_path).to(device)

        if pred_img.shape[-2:] != gt_img.shape[-2:]:
            gt_img = TF.resize(gt_img, pred_img.shape[-2:])

        with torch.no_grad():
            d = lpips_fn(pred_img, gt_img)
        lpips_values.append(float(d.item()))

if lpips_values:
    lpips_values = np.array(lpips_values)
    print(f"pairs count: {len(lpips_values)}")
    print(f"LPIPS mean: {lpips_values.mean():.6f}")
else:
    print("no pair images")

if missing_pairs:
    print(f"\n have {len(missing_pairs)} cannot have pair")
    print(missing_pairs[:10])


In [ ]:
from cleanfid import fid
fid_val = fid.compute_fid(Test_DIR_dst, Refine_DIR, mode="clean")
print(f"FID = {fid_val:.4f}")

In [ ]:
import os
import numpy as np
from PIL import Image
import torchvision.transforms.functional as TF
from skimage.metrics import structural_similarity as ssim

IMG_EXTS = ('.png', '.jpg', '.jpeg', '.bmp', '.webp')
Test_NoMask_DIR  = Test_DIR_dst

def load_img_np(path, size=None):
    img = Image.open(path).convert('RGB')
    if size is not None:
        img = img.resize(size, Image.BICUBIC)
    return np.array(img, dtype=np.uint8)

def calc_psnr(img1, img2):
    mse = np.mean((img1.astype(np.float32) - img2.astype(np.float32)) ** 2)
    if mse == 0:
        return 100.0
    PIXEL_MAX = 255.0
    return 20 * np.log10(PIXEL_MAX / np.sqrt(mse))

def calc_ssim(img1, img2):
    img1_gray = np.dot(img1[..., :3], [0.299, 0.587, 0.114])
    img2_gray = np.dot(img2[..., :3], [0.299, 0.587, 0.114])
    return ssim(img1_gray, img2_gray, data_range=255)

gt_dict = {}
for fname in os.listdir(Test_NoMask_DIR):
    if not fname.lower().endswith(IMG_EXTS):
        continue
    key = os.path.splitext(fname)[0]
    gt_dict[key] = os.path.join(Test_NoMask_DIR, fname)

print(f"GT image number: {len(gt_dict)}")

psnr_values = []
ssim_values = []
missing_pairs = []

for root, _, files in os.walk(Refine_DIR):
    for fname in files:
        if not fname.lower().endswith(IMG_EXTS):
            continue

        pred_path = os.path.join(root, fname)
        pred_key  = os.path.splitext(fname)[0]

        gt_key = pred_key.replace('_mask000', '').replace('_mask', '')

        if gt_key not in gt_dict:
            missing_pairs.append(fname)
            continue

        gt_path = gt_dict[gt_key]

        pred_np = load_img_np(pred_path)
        gt_np   = load_img_np(gt_path)

        if pred_np.shape[:2] != gt_np.shape[:2]:
            h, w = pred_np.shape[:2]
            gt_np = load_img_np(gt_path, size=(w, h))

        psnr_score = calc_psnr(pred_np, gt_np)
        ssim_score = calc_ssim(pred_np, gt_np)

        psnr_values.append(psnr_score)
        ssim_values.append(ssim_score)

if psnr_values:
    psnr_values = np.array(psnr_values)
    ssim_values = np.array(ssim_values)
    print(f"pairs count: {len(psnr_values)}")
    print(f"PSNR mean: {psnr_values.mean():.6f}")
    print(f"SSIM mean: {ssim_values.mean():.6f}")
else:
    print("no pair images")

if missing_pairs:
    print(f"\n have {len(missing_pairs)} cannot have pair")
    print(missing_pairs[:10])
